# Imports

In [1]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
from scipy.stats import t
import dask

In [2]:
dask.config.set({'array.chunk-size': '8 GiB'})
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='32GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='01:00:00', # Amount of wall time
)
cluster.scale(jobs=4)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34897 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/34897/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/34897/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.206:40989,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/34897/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [3]:
dask.config.set({'array.chunk-size': '8 GiB'})
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='32GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='01:00:00', # Amount of wall time
)
cluster.scale(jobs=4)
client = Client(cluster)
client

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 40861 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/40861/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/40861/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.206:41647,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/40861/status,Total threads: 0
Started: Just now,Total memory: 0 B


# function

In [4]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Import Data

In [5]:
EOF_ds = xr.open_dataset("/glade/work/acruz/E3SMv2LE/EANI_CANI_E3SMv2.nc", chunks='auto')
EOF_ds

<xarray.Dataset> Size: 436kB
Dimensions:  (member: 21, time: 1212)
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    lev      (time) float64 10kB dask.array<chunksize=(1212,), meta=np.ndarray>
    month    (time) int64 10kB dask.array<chunksize=(1212,), meta=np.ndarray>
Data variables:
    EANI     (member, time) float64 204kB dask.array<chunksize=(21, 1212), meta=np.ndarray>
    CANI     (member, time) float64 204kB dask.array<chunksize=(21, 1212), meta=np.ndarray>
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

In [6]:
ssta_ds = xr.open_dataset('/glade/work/acruz/E3SMv2LE/SST_anom_hist.nc')
ssta_ds.coords['lon'] = (ssta_ds.coords['lon'] + 180) % 360 - 180
ssta_ds = ssta_ds.sortby(ssta_ds.lon)
ssta_ds = ssta_ds.assign_coords({'member': ssta_ds['member']})
ssta_ds

<xarray.Dataset> Size: 9GB
Dimensions:  (member: 21, time: 1980, lat: 192, lon: 288)
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * time     (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-01 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
    lev      (time) float64 16kB ...
    month    (time) int64 16kB ...
Data variables:
    T        (member, time, lat, lon) float32 9GB ...

# Data selection

In [7]:
dates = slice('1914-01-01', '2014-12-31')
ssta_ds = ssta_ds.sel(time=dates)
EOF_ds = EOF_ds.sel(time=dates)
EOF_ds

<xarray.Dataset> Size: 436kB
Dimensions:  (member: 21, time: 1212)
Coordinates:
  * member   (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    lev      (time) float64 10kB dask.array<chunksize=(1212,), meta=np.ndarray>
    month    (time) int64 10kB dask.array<chunksize=(1212,), meta=np.ndarray>
Data variables:
    EANI     (member, time) float64 204kB dask.array<chunksize=(21, 1212), meta=np.ndarray>
    CANI     (member, time) float64 204kB dask.array<chunksize=(21, 1212), meta=np.ndarray>
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      2025-07-14

In [8]:
JJAds = ssta_ds['T'].sel(time=ssta_ds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
SONds = ssta_ds['T'].sel(time=ssta_ds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()

EANI = EOF_ds['EANI'].sel(time=EOF_ds['EANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
CANI = EOF_ds['CANI'].sel(time=EOF_ds['CANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

# Regressions

In [9]:
JJA_EANI = xr_regression(EANI, JJAds, dim='year')
JJA_CANI = xr_regression(CANI, JJAds, dim='year')
SON_EANI = xr_regression(EANI, SONds, dim='year')
SON_CANI = xr_regression(CANI, SONds, dim='year')

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/xarray/computation/apply_ufunc.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 469.70 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/numpy/lib/_nanfunctions_impl.py:2015

In [10]:
JJA_EANI

<xarray.Dataset> Size: 35MB
Dimensions:    (member: 21, lat: 192, lon: 288)
Coordinates:
  * member     (member) int64 168B 0 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
  * lat        (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon        (lon) float64 2kB -180.0 -178.8 -177.5 ... 176.2 177.5 178.8
Data variables:
    cov        (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    cor        (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    r2         (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    slope      (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    intercept  (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    pvalue     (member, lat, lon) float32 5MB nan nan nan nan ... nan nan nan
    stderr     (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    n          (member, lat, lon) int16 2MB 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0

In [11]:
regression = xr.Dataset(
    data_vars={
        "JJA_SSTA_EANI_slope": (['member', 'lat', 'lon'], JJA_EANI['slope'].data),
        "JJA_SSTA_EANI_pval": (['member', 'lat', 'lon'], JJA_EANI['pvalue'].data),
        "JJA_SSTA_CANI_slope": (['member', 'lat', 'lon'], JJA_CANI['slope'].data),
        "JJA_SSTA_CANI_pval": (['member', 'lat', 'lon'], JJA_CANI['pvalue'].data),
        "SON_SSTA_EANI_slope": (['member', 'lat', 'lon'], SON_EANI['slope'].data),
        "SON_SSTA_EANI_pval": (['member', 'lat', 'lon'], SON_EANI['pvalue'].data),
        "SON_SSTA_CANI_slope": (['member', 'lat', 'lon'], SON_CANI['slope'].data),
        "SON_SSTA_CANI_pval": (['member', 'lat', 'lon'], SON_CANI['pvalue'].data)
    },
    coords={
        "member": JJA_EANI['member'].data,
        "lat": JJA_EANI['lat'].data,
        "lon": JJA_EANI['lon'].data,
    },
    attrs={
        "Description": "Linear regression results from E3SM SST anomaalies between 1914-2014"
    }
)
regression

<xarray.Dataset> Size: 37MB
Dimensions:              (member: 21, lat: 192, lon: 288)
Coordinates:
  * member               (member) int64 168B 0 1 2 3 4 5 6 ... 15 16 17 18 19 20
  * lat                  (lat) float64 2kB -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                  (lon) float64 2kB -180.0 -178.8 -177.5 ... 177.5 178.8
Data variables:
    JJA_SSTA_EANI_slope  (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    JJA_SSTA_EANI_pval   (member, lat, lon) float32 5MB nan nan nan ... nan nan
    JJA_SSTA_CANI_slope  (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    JJA_SSTA_CANI_pval   (member, lat, lon) float32 5MB nan nan nan ... nan nan
    SON_SSTA_EANI_slope  (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    SON_SSTA_EANI_pval   (member, lat, lon) float32 5MB nan nan nan ... nan nan
    SON_SSTA_CANI_slope  (member, lat, lon) float32 5MB dask.array<chunksize=(21, 192, 288), meta=np.ndarray>
    SON_SSTA_CANI_pval   (member, lat, lon) float32 5MB nan nan nan ... nan nan
Attributes:
    Description:  Linear regression results from E3SM SST anomaalies between ...

# Export

In [12]:
regression.to_zarr('/glade/work/acruz/E3SMv2LE/Regressions/SSTA', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/distributed/client.py:3374: UserWarning: Sending large graph of size 912.84 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


In [13]:
client.shutdown()